In [43]:
import polars as pl
import glob
import os

In [44]:
DIR='../outputs.mapping/cds3/outputs.singleclust'

In [49]:
template = '../outputs.mapping/cds3/outputs.singleclust/{metag}.x.{species}.depth.txt'

def read_depth(metag, species):
    filename = template.format(metag=metag, species=species)
    df = pl.read_csv(filename, separator='\t', has_header=False,
                 new_columns=('gene', 'pos', 'cov', 'foo'))
    sum_df = df.group_by('gene').agg(
        ((pl.col("cov") > 0).sum() / pl.col("pos").len()).alias("breadth")
    ).with_columns(
        (pl.lit(metag).alias("metag")),
        (pl.lit(species).alias("species"))
)
    return sum_df

read_depth('ERR1135199', 's__Cryptobacteroides sp900546925')

gene,breadth,metag,species
str,f64,str,str
"""CPBNAKKL_01317""",0.461686,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""PGBAIDML_01792""",0.0,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""FNIGJHLI_00487""",0.0,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""FNIGJHLI_00528""",0.506611,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""BEKEPKDC_01021""",0.702721,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
…,…,…,…
"""BDFOMKKK_01862""",0.105691,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""LHAJOBLP_00025""",0.933333,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"
"""IEFFFPKJ_00801""",0.546296,"""ERR1135199""","""s__Cryptobacteroides sp9005469…"


In [47]:
filename = 'ERR1135199.x.s__Cryptobacteroides sp900546925.depth.txt'
filenames = glob.glob(f"{DIR}/*.depth.txt")

dflist = []
for i, n in enumerate(filenames):
    if i % 100 == 0:
        print(f"{i} of {len(filenames)}")
    n = os.path.basename(n)
    metag, _, species, _ = n.split('.', 3)
    dflist.append(read_depth(metag, species))



0 1400
100 1400
200 1400
300 1400
400 1400
500 1400
600 1400
700 1400
800 1400
900 1400
1000 1400
1100 1400
1200 1400
1300 1400


NameError: name 'pd' is not defined

In [48]:
df = pl.concat(dflist)

In [50]:
df

gene,breadth,metag,species
str,f64,str,str
"""BCJOJFGP_00839""",0.998889,"""ERR8314788""","""s__Prevotella sp000434975"""
"""CHAJGCIB_01628""",1.0,"""ERR8314788""","""s__Prevotella sp000434975"""
"""PJFPEDHO_01064""",0.858641,"""ERR8314788""","""s__Prevotella sp000434975"""
"""ABGMHDFK_01758""",0.974052,"""ERR8314788""","""s__Prevotella sp000434975"""
"""LPDFMKHA_01513""",0.956522,"""ERR8314788""","""s__Prevotella sp000434975"""
…,…,…,…
"""AKLOHKAG_01416""",0.193215,"""SRR8960391""","""s__Mogibacterium_A kristiansen…"
"""NADIOELI_00170""",0.0,"""SRR8960391""","""s__Mogibacterium_A kristiansen…"
"""IEPMJGAI_00771""",1.0,"""SRR8960391""","""s__Mogibacterium_A kristiansen…"


In [57]:
agg_df = df.group_by(['species', 'gene']).agg(
    ((pl.col("breadth") > 0.1).sum() / pl.col("breadth").len()).alias("f")
)
agg_df

species,gene,f
str,str,f64
"""s__Prevotella sp000434975""","""OPNPMHKL_01111""",0.82
"""s__Holdemanella porci""","""CAJDFOMO_00564""",0.74
"""s__UBA2868 sp004552595""","""HEOOOEGB_00722""",0.61
"""s__Sodaliphilus sp004557565""","""AOJPJKKJ_01881""",0.91
"""s__Gemmiger qucibialis""","""OMLAPDIE_01597""",0.27
…,…,…
"""s__Prevotella sp000434975""","""FFDABIAA_00367""",0.7
"""s__Sodaliphilus sp004557565""","""JFIMJEBI_00808""",0.91
"""s__Prevotella sp000434975""","""LPDFMKHA_01233""",0.71


In [64]:
for species in agg_df['species'].unique():
    print(species)
    foo_df = agg_df.filter(pl.col("species") == species)
    print(foo_df.sort(by='f').filter(pl.col('f') > 0.8))

s__Bariatricus sp004560705
shape: (22, 3)
┌────────────────────────────┬────────────────┬──────┐
│ species                    ┆ gene           ┆ f    │
│ ---                        ┆ ---            ┆ ---  │
│ str                        ┆ str            ┆ f64  │
╞════════════════════════════╪════════════════╪══════╡
│ s__Bariatricus sp004560705 ┆ ELOEBFJM_01459 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ KJMAJKMB_02318 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ LNGHPPMH_01876 ┆ 0.81 │
│ s__Bariatricus sp004560705 ┆ EMNHIHKA_01693 ┆ 0.82 │
│ s__Bariatricus sp004560705 ┆ LNGHPPMH_00679 ┆ 0.83 │
│ …                          ┆ …              ┆ …    │
│ s__Bariatricus sp004560705 ┆ AIEIMHOC_00182 ┆ 0.89 │
│ s__Bariatricus sp004560705 ┆ FMMMFAAO_00746 ┆ 0.89 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_02301 ┆ 0.93 │
│ s__Bariatricus sp004560705 ┆ LEHFNBLN_00701 ┆ 0.95 │
│ s__Bariatricus sp004560705 ┆ CHOLCMOC_00863 ┆ 0.97 │
└────────────────────────────┴────────────────┴──────┘
s__Mogibacterium_A kris

In [70]:
df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth')

gene,breadth,metag,species
str,f64,str,str
"""EHOAPHDI_01174""",0.0,"""SRR12795775""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",0.0,"""SRR8655118""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",0.0,"""ERR3212024""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",0.0,"""SRR5240729""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",0.0,"""SRR12795790""","""s__Phascolarctobacterium_A suc…"
…,…,…,…
"""EHOAPHDI_01174""",1.0,"""ERR1135331""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",1.0,"""SRR11124923""","""s__Phascolarctobacterium_A suc…"
"""EHOAPHDI_01174""",1.0,"""SRR11489769""","""s__Phascolarctobacterium_A suc…"


In [71]:
df.filter((pl.col("gene") == 'EHOAPHDI_01174')).sort(by='breadth').filter(pl.col("metag") == "ERR1135199")

gene,breadth,metag,species
str,f64,str,str
"""EHOAPHDI_01174""",0.279642,"""ERR1135199""","""s__Phascolarctobacterium_A suc…"
